# Model Selection and Hyperparameter Tuning

## Learning Objectives
- Understand the importance of model selection in machine learning
- Learn various techniques for hyperparameter tuning
- Master Grid Search and Random Search methods
- Practice with cross-validation for robust model evaluation
- Compare different models systematically

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_wine, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold, validation_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# For better visualization
%matplotlib inline

## 1. Understanding Model Selection

Model selection involves choosing the best algorithm and its hyperparameters for a given problem. Let's start by comparing several classifiers.

In [ ]:
# Load the Wine dataset for classification
wine = load_wine()
X = wine.data
y = wine.target

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dataset Information:")
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"Number of features: {X_train.shape[1]}")
print(f"Number of classes: {len(np.unique(y))}")

In [ ]:
# Define multiple classifiers
classifiers = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Support Vector Machine': SVC(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

# Evaluate each classifier using cross-validation
cv_scores = {}
cv_std = {}

for name, clf in classifiers.items():
    # Perform 5-fold cross-validation
    scores = cross_val_score(clf, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores[name] = scores.mean()
    cv_std[name] = scores.std()
    
    print(f"{name}:")
    print(f"  CV Accuracy: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")
    print(f"  Individual scores: {scores}\n")

In [ ]:
# Visualize cross-validation results
plt.figure(figsize=(12, 8))
names = list(cv_scores.keys())
means = list(cv_scores.values())
stds = list(cv_std.values())

y_pos = np.arange(len(names))
plt.barh(y_pos, means, xerr=stds, align='center', alpha=0.7)
plt.yticks(y_pos, names)
plt.xlabel('Cross-Validation Accuracy')
plt.title('Classifier Comparison with Cross-Validation')
plt.xlim(0, 1.1)
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Grid Search for Hyperparameter Tuning

Grid Search exhaustively searches through a specified parameter grid to find the best hyperparameters.

In [ ]:
# Perform Grid Search on the best performing classifier
# Let's use Random Forest as an example
rf = RandomForestClassifier(random_state=42)

# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print("Parameter grid for Random Forest:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")
print(f"\nTotal combinations: {np.prod([len(v) for v in param_grid.values()])}")

In [ ]:
# Perform Grid Search with cross-validation
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,  # Use all available cores
    verbose=1
)

# Fit the grid search
print("Performing Grid Search...")
grid_search.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate the best model on the test set
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_pred)

print(f"Test set accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=wine.target_names))

## 3. Random Search for Hyperparameter Tuning

Random Search samples a fixed number of parameter settings from specified distributions, which can be more efficient than Grid Search.

In [ ]:
# Define parameter distributions for Random Search
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': [None] + list(randint(10, 50).rvs(5)),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['auto', 'sqrt', 'log2', None]
}

print("Parameter distributions for Random Search:")
for param, dist in param_dist.items():
    print(f"  {param}: {dist}")

In [ ]:
# Perform Random Search
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=50,  # Number of parameter settings sampled
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Fit the random search
print("Performing Random Search...")
random_search.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {random_search.best_params_}")
print(f"Best cross-validation score: {random_search.best_score_:.4f}")

In [ ]:
# Compare Grid Search and Random Search results
random_best = random_search.best_estimator_
y_pred_random = random_best.predict(X_test_scaled)
test_accuracy_random = accuracy_score(y_test, y_pred_random)

print("Comparison of Grid Search vs Random Search:")
print(f"Grid Search - Test accuracy: {test_accuracy:.4f}")
print(f"Random Search - Test accuracy: {test_accuracy_random:.4f}")
print(f"Difference: {abs(test_accuracy - test_accuracy_random):.4f}")

## 4. Validation Curves

Validation curves help us understand how model performance changes with different hyperparameter values.

In [ ]:
# Create validation curves for a specific hyperparameter
# Let's use n_estimators for Random Forest
param_range = [10, 50, 100, 150, 200, 250, 300]

train_scores, test_scores = validation_curve(
    RandomForestClassifier(random_state=42),
    X_train_scaled, y_train,
    param_name='n_estimators',
    param_range=param_range,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Calculate mean and standard deviation
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

In [ ]:
# Plot validation curves
plt.figure(figsize=(10, 6))
plt.plot(param_range, train_mean, 'o-', color='blue', label='Training score')
plt.fill_between(param_range, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
plt.plot(param_range, test_mean, 'o-', color='red', label='Cross-validation score')
plt.fill_between(param_range, test_mean - test_std, test_mean + test_std, alpha=0.1, color='red')
plt.xlabel('Number of Estimators')
plt.ylabel('Accuracy')
plt.title('Validation Curve for Random Forest (n_estimators)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Pipeline with Hyperparameter Tuning

Combining preprocessing and model selection in a single pipeline.

In [ ]:
# Create a pipeline that includes preprocessing and model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(random_state=42))
])

print("Pipeline created successfully!")
print("Pipeline steps:")
for name, step in pipeline.steps:
    print(f"  - {name}: {step}")

In [ ]:
# Define parameter grid for pipeline
# Note: parameters need to be prefixed with the step name
pipe_param_grid = [
    {
        'classifier': [SVC(random_state=42)],
        'classifier__C': [0.1, 1, 10, 100],
        'classifier__kernel': ['linear', 'rbf'],
        'classifier__gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
    },
    {
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [None, 10, 20]
    }
]

print("Pipeline parameter grid:")
for i, param_set in enumerate(pipe_param_grid):
    print(f"  Set {i+1}:")
    for param, values in param_set.items():
        print(f"    {param}: {values}")

In [ ]:
# Perform Grid Search on the pipeline
pipe_grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=pipe_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Fit the pipeline grid search
print("Performing Grid Search on Pipeline...")
pipe_grid_search.fit(X_train, y_train)  # Note: using unscaled X_train since scaler is in pipeline

print(f"\nBest parameters: {pipe_grid_search.best_params_}")
print(f"Best cross-validation score: {pipe_grid_search.best_score_:.4f}")

In [ ]:
# Evaluate the best pipeline on the test set
best_pipeline = pipe_grid_search.best_estimator_
y_pred_pipe = best_pipeline.predict(X_test)  # Note: using unscaled X_test
test_accuracy_pipe = accuracy_score(y_test, y_pred_pipe)

print(f"Best pipeline test accuracy: {test_accuracy_pipe:.4f}")
print("\nBest pipeline steps:")
for name, step in best_pipeline.steps:
    print(f"  - {name}: {step}")

## 6. Advanced Model Selection Techniques

Let's explore some advanced techniques for model selection.

In [ ]:
# Nested Cross-Validation for unbiased performance estimation
from sklearn.model_selection import cross_validate

# Define a model with hyperparameter tuning
nested_model = RandomForestClassifier(random_state=42)
nested_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20]
}

# Inner loop: Grid Search for hyperparameter tuning
inner_cv = GridSearchCV(
    estimator=nested_model,
    param_grid=nested_param_grid,
    cv=3,  # Inner cross-validation folds
    scoring='accuracy'
)

# Outer loop: Cross-validation for performance estimation
outer_cv_scores = cross_val_score(inner_cv, X_train_scaled, y_train, cv=5, scoring='accuracy')

print("Nested Cross-Validation Results:")
print(f"Individual scores: {outer_cv_scores}")
print(f"Mean accuracy: {outer_cv_scores.mean():.4f} (+/- {outer_cv_scores.std() * 2:.4f})")

In [ ]:
# Model comparison with statistical significance testing
from scipy import stats

# Perform cross-validation for multiple models
models_to_compare = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42)
}

# Collect cross-validation scores
cv_results = {}
for name, model in models_to_compare.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

In [ ]:
# Statistical comparison using paired t-test
print("\nStatistical Significance Tests (p-values):")
model_names = list(cv_results.keys())
for i in range(len(model_names)):
    for j in range(i+1, len(model_names)):
        model1, model2 = model_names[i], model_names[j]
        t_stat, p_val = stats.ttest_rel(cv_results[model1], cv_results[model2])
        print(f"{model1} vs {model2}: p = {p_val:.4f}")
        if p_val < 0.05:
            print(f"  -> Significant difference (p < 0.05)")
        else:
            print(f"  -> No significant difference (p >= 0.05)")

## 7. Practical Tips for Model Selection

Let's summarize some practical tips for effective model selection.

In [ ]:
# Create a summary of best practices
best_practices = [
    "1. Always use cross-validation for model evaluation",
    "2. Split your data into train/validation/test sets",
    "3. Use Grid Search for small parameter spaces, Random Search for large ones",
    "4. Consider nested cross-validation for unbiased performance estimates",
    "5. Don't forget to scale your data when needed",
    "6. Use pipelines to prevent data leakage",
    "7. Consider multiple metrics, not just accuracy",
    "8. Validate statistical significance of performance differences",
    "9. Document your model selection process",
    "10. Retrain your final model on the full dataset"
]

print("Best Practices for Model Selection and Hyperparameter Tuning:")
for practice in best_practices:
    print(practice)

In [ ]:
# Final example: Complete workflow
print("=== Complete Model Selection Workflow ===")
print("\nStep 1: Load and explore data")
print("Step 2: Split data into train/validation/test sets")
print("Step 3: Preprocess data (scaling, encoding, etc.)")
print("Step 4: Select candidate models")
print("Step 5: Define parameter grids for each model")
print("Step 6: Perform hyperparameter tuning with cross-validation")
print("Step 7: Compare models using statistical tests")
print("Step 8: Select the best model and parameters")
print("Step 9: Evaluate final model on test set")
print("Step 10: Document results and save the model")

## Summary

In this notebook, we've covered:

1. **Model Selection**: Comparing different algorithms using cross-validation
2. **Grid Search**: Exhaustive search through parameter combinations
3. **Random Search**: Efficient sampling of parameter space
4. **Validation Curves**: Understanding model behavior with different parameters
5. **Pipeline Integration**: Combining preprocessing and hyperparameter tuning
6. **Advanced Techniques**: Nested cross-validation and statistical significance testing
7. **Best Practices**: Guidelines for effective model selection

Key takeaways:
- Model selection is a critical step in the ML pipeline
- Cross-validation provides robust performance estimates
- Grid Search is thorough but can be computationally expensive
- Random Search is often more efficient for large parameter spaces
- Pipelines help prevent data leakage and streamline workflows
- Statistical significance testing helps validate performance differences

Mastering these techniques will help you build more robust and reliable machine learning models.